# Lab 05: Structured Outputs and JSON Reliability (Gemini 3.1)

Predictable output is the backbone of AI agents and automated workflows. In this lab, we will explore advanced **JSON generation** with **Gemini 3.1 Flash-Lite**, focusing on complex schemas, Enums, and data extraction.

## Objectives
1. Enforce complex **nested Pydantic schemas**.
2. Use **Enums** to restrict model choices.
3. Extract structured data from messy, unstructured text.
4. Handle and parse JSON responses safely.

In [ ]:
import os
from enum import StrEnum

from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Client initialized.")

## 1. Complex Nested Schemas and Enums

Gemini 3.1 models can follow deeply nested structures. We will define a schema for a "Support Ticket" system that includes categories and priority levels.

In [ ]:
class TicketPriority(StrEnum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    URGENT = "urgent"

class TicketCategory(StrEnum):
    BILLING = "billing"
    TECHNICAL = "technical"
    FEATURE_REQUEST = "feature_request"
    OTHER = "other"

class SupportTicket(BaseModel):
    summary: str = Field(description="A 5-word summary of the issue")
    priority: TicketPriority
    category: TicketCategory
    affected_systems: list[str]
    is_resolved_by_faq: bool
    estimated_wait_time_minutes: int | None

prompt = "My laptop is overheating whenever I run the Chrome browser, " \
         "and the fan makes a loud noise."\
         "I checked the FAQ but found no help. I'm a premium user."

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=SupportTicket,
    ),
    contents=prompt
)

# Parse and display
ticket = SupportTicket.model_validate_json(response.text)
print("Structured Ticket Output:")
print(ticket.model_dump_json(indent=2))

## 2. Extraction from Unstructured Text

Structured output is perfect for data extraction. Let's extract multiple items and their attributes from a messy receipt-like text.

In [ ]:
class InvoiceItem(BaseModel):
    description: str
    unit_price: float
    quantity: int
    total: float

class Invoice(BaseModel):
    merchant: str
    date: str
    items: list[InvoiceItem]
    tax_amount: float
    grand_total: float

raw_text = """
Yesterday at Joe's Pizza (March 12th, 2026), I bought two large Pepperoni pizzas
for $15.50 each. I also got a 2L soda for $3.25. The tax was $2.50.
Overall I paid $36.75.
"""

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Invoice,
    ),
    contents=f"Extract invoice data from this text: {raw_text}"
)

display(Markdown(f"```json\n{response.text}\n```"))

## 3. Schema Enforcement with Reasoning

You can force the model to "think" inside the JSON by adding a `reasoning` or `explanation` field to the schema. This often improves the accuracy of the other fields.

In [ ]:
class SentimentAnalysis(BaseModel):
    sentiment_score: float = Field(
        description="Score from -1.0 (very negative) to 1.0 (very positive)"
    )
    explanation: str = Field(description="Detailed reasoning for this score")
    key_phrases: list[str]

prompt = "I'm extremely disappointed with the new update. It's slow and keeps " \
         "crashing, though the UI looks nice."

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=SentimentAnalysis,
    ),
    contents=prompt
)

print(response.text)

## Summary

In this lab, you explored the reliability of Gemini 3.1 for structured data:
1. **Pydantic integration** for automatic schema generation and validation.
2. **Enum support** to ensure strict adherence to categories.
3. **Data extraction** patterns for converting text to databases.
4. **Reasoning injection** inside JSON to boost extraction logic.